# PASTIS-R: Benchmark Inspection

PASTIS-R is the dense crop-type segmentation benchmark. One raw patch is 128x128 pixels with Sentinel-2, Sentinel-1, per-pixel labels, and a published fold id. The loader exposes it lazily as 64x64 tiles.


## Setup


In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from dataio.get_input import get_input
from dataio.inspection import benchmark_input_contract_table, benchmark_metadata_table

REPO = Path.cwd()
while not (REPO / "src").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "src"))

BENCHMARK_ROOT = REPO / "data" / "input" / "benchmarks"
print("repo:", REPO)
print("benchmark root:", BENCHMARK_ROOT)

## Load Benchmark


In [ ]:

MAX_SAMPLES = 8
LOAD_KWARGS = {}

bench = None
load_error = None
try:
    bench = get_input("pastis", root=BENCHMARK_ROOT, max_samples=MAX_SAMPLES, shuffle=False, **LOAD_KWARGS)
    print(f"loaded {bench.name}: {bench.n_samples} samples")
except Exception as exc:
    load_error = exc
    print(f"benchmark could not be loaded locally: {type(exc).__name__}: {exc}")

## Input Contract and Available Metadata


In [ ]:
if bench is None:
    display(Markdown(f"Benchmark tables are skipped because the local data could not be read: `{type(load_error).__name__}: {load_error}`"))
else:
    display(Markdown("### Model-facing input contract"))
    display(benchmark_input_contract_table(bench))
    display(Markdown("### Other benchmark metadata available"))
    display(benchmark_metadata_table(bench))

## Lazy Tile Inspection


In [ ]:
if bench is not None:
    rows = []
    for patch in bench.patches[:8]:
        rows.append({
            "patch_id": patch.patch_id,
            "fold": patch.fold,
            "s2_observations": len(patch.s2_months),
            "s1_observations": len(patch.s1_months),
            "s2_months": list(map(int, patch.s2_months[:8])),
            "s1_months": list(map(int, patch.s1_months[:8])),
        })
    display(pd.DataFrame(rows))

    tile_id, tile_fold, tile, labels = next(bench.iter_tiles())
    print("first tile:", tile_id, "fold", tile_fold)
    print("s2", tile.s2.shape, "s1", tile.s1.shape, "labels", tile.labels.shape)
    print("valid pixels", int(tile.valid.sum()), "/", tile.valid.size)
else:
    print("PASTIS-R tile inspection skipped")

## Takeaways

The repository key/path is `pastis`; the dataset name remains PASTIS-R in prose. Fold ids are the central non-input metadata: random_id uses the published train/val/test assignment, while geographic_ood holds out spatial folds.
